# Lesson 8: Datasets and DataLoaders

A network learns from data, and real datasets are too big to hand over all at once. This lesson is about how PyTorch feeds data to a network: a **Dataset** holds the examples, and a **DataLoader** serves them in small, shuffled **batches**. You will load the real MNIST handwriting dataset and get it ready to train on. No training yet, that comes in the next lessons.

Run every cell top to bottom, then do the exercises at the end and upload this notebook for feedback and a grade.

## Setup

In [ ]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

## MNIST via torchvision

MNIST is 70,000 small grayscale images of handwritten digits 0-9, each 28x28 pixels, split into 60,000 for training and 10,000 for testing. `torchvision` downloads and loads it for you. The first run downloads about 11 MB into a local `data` folder, once; after that it loads from disk.

In [ ]:
transform = transforms.ToTensor()   # each image becomes a (1, 28, 28) float tensor in [0, 1]

train_ds = datasets.MNIST(root='./data', train=True,  download=True, transform=transform)
test_ds  = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

print(len(train_ds), 'training images')   # 60000
print(len(test_ds),  'test images')       # 10000

## One example

A Dataset is indexable: `train_ds[i]` gives the i-th (image, label) pair. Each image is a `(1, 28, 28)` tensor (one grayscale channel, 28 rows, 28 columns) and the label is the digit it shows.

In [ ]:
image, label = train_ds[0]
print('image shape:', image.shape)   # torch.Size([1, 28, 28])
print('label:', label)               # the digit

plt.imshow(image.squeeze(), cmap='gray')
plt.title(f'label: {label}')
plt.axis('off')
plt.show()

## Why batches, and the DataLoader

Feeding one image at a time is slow and noisy; feeding all 60,000 at once may not fit in memory. The sweet spot is a **mini-batch**, a small group (say 64) processed together. A **DataLoader** wraps a Dataset and hands out batches, shuffled if you ask. Shuffling the training data each pass keeps the network from memorizing the order.

In [ ]:
train_loader = DataLoader(train_ds, batch_size=64,   shuffle=True)
test_loader  = DataLoader(test_ds,  batch_size=1000, shuffle=False)

images, labels = next(iter(train_loader))
print('images:', images.shape)   # torch.Size([64, 1, 28, 28])  (batch, channel, H, W)
print('labels:', labels.shape)   # torch.Size([64])

The DataLoader stacked 64 images into one tensor with a new batch axis at the front. This is the `(batch, ...)` shape you have tracked since Lesson 2.

## Flattening for an MLP

An MLP expects a flat vector per example, not a 28x28 grid. Flatten each image from `(1, 28, 28)` into a 784-length vector (28 * 28 = 784), keeping the batch axis.

In [ ]:
x = images.view(images.size(0), -1)   # keep the batch axis, flatten the rest
print(x.shape)   # torch.Size([64, 784])

Now each of the 64 images is a 784-number vector, exactly what `nn.Linear(784, ...)` wants. (For the transformer you will keep the structure instead of flattening, but for a plain MLP on MNIST, flat is right.)

## Epochs and batches, the vocabulary

Two words you will use constantly next:

- a **batch** is one group of examples processed together (64 images here),
- an **epoch** is one full pass through the entire dataset (all 60,000 images, batch by batch).

Training runs for several epochs, and the DataLoader reshuffles the batches at the start of each one. With batch size 64, one epoch of MNIST is 60000 / 64 = 938 batches.

## Exercises

Do these in the cells below, then upload the notebook. Only your code under each `# YOUR CODE HERE` line is graded.

In [ ]:
# E1: Load the MNIST TEST set with torchvision: datasets.MNIST(root='./data', train=False,
#     download=True, transform=transforms.ToTensor()). Print how many images it has (expect 10000).
# YOUR CODE HERE (write your answer below)


In [ ]:
# E2: Make a DataLoader for that test set with batch_size=1000 and shuffle=False. Grab one
#     batch with next(iter(...)) and print the images shape and the labels shape
#     (expect (1000, 1, 28, 28) and (1000,)).
# YOUR CODE HERE (write your answer below)


In [ ]:
# E3: Flatten a batch. Make a fake batch of images with torch.randn(64, 1, 28, 28), flatten it
#     to shape (64, 784) with .view (keep the batch axis), and print the resulting shape.
# YOUR CODE HERE (write your answer below)


In [ ]:
# E4: Index a dataset. From the MNIST test set you loaded in E1, take the example at index 0,
#     and print the image's shape and its label.
# YOUR CODE HERE (write your answer below)
